# Tahap 2: Embedding (SBERT) & Indexing (ChromaDB) — Chatbot Rekomendasi Resep

Lanjutan dari `preprocessing.ipynb`. Notebook ini:
1. Memuat dataset hasil preprocessing (5000 resep sampel)
2. Meng-encode `embedding_text` tiap resep pakai SBERT **`LazarusNLP/all-indo-e5-small-v4`**
3. Indexing hasil embedding ke **ChromaDB**
4. Uji coba retrieval sederhana (tanpa reranking efisiensi bahan dulu — itu tahap berikutnya)

## ⚠️ Catatan penting sebelum mulai

**1) Bug di file CSV hasil preprocessing**

File `dataset_resep_preprocessed_5000.csv` yang di-generate dari cell terakhir `preprocessing.ipynb`:
```python
df = pd.read_parquet('dataset_resep_preprocessed_5000.parquet')
df.to_csv('dataset_resep_preprocessed_5000.csv', index=False)
```
punya bug pada kolom-kolom bertipe list/dict (`ingredients_clean`, `ingredients_parsed`, `ingredients_name_only`, `steps`).

**Penyebabnya:** waktu file parquet dibaca ulang dengan `pd.read_parquet`, kolom-kolom list itu jadi **numpy array**, bukan Python list biasa. Waktu numpy array itu di-`.to_csv()`, pandas pakai `str(array)` yang formatnya **tidak pakai koma** antar elemen, contoh:
```
['1 ekor Baby Tuna' '1 sdm Asam Jawa' '1 sdt Garam' ...]
```
Ini **kelihatannya** seperti list Python valid, tapi kalau nanti dibaca ulang pakai `ast.literal_eval()`, Python akan otomatis **menggabungkan string yang bersebelahan** (adjacent string literal concatenation) jadi SATU string panjang, bukan list berisi banyak item. Bug ini *silent* — tidak error, tapi datanya jadi rusak/salah tanpa ketahuan.

Sudah saya cek ke 5000 baris x 4 kolom tersebut → semuanya kena bug ini kalau di-`ast.literal_eval()` langsung.

**Solusinya di notebook ini:**
- **Opsi A (disarankan):** load langsung dari file **`.parquet`**-nya (bukan CSV). Parquet menyimpan list/dict secara native lewat pyarrow, jadi tidak ada masalah round-trip ini sama sekali.
- **Opsi B (fallback):** kalau cuma punya file CSV-nya, notebook ini otomatis "memperbaiki" string yang rusak itu pakai fungsi `repair_numpy_list_str()` di bawah (regex untuk menyisipkan koma di antara elemen, lalu `ast.literal_eval`). Sudah divalidasi 100% cocok jumlahnya dengan `num_ingredients` (5000/5000 baris cocok), dan 99.4% cocok untuk `steps` (28/5000 baris meleset karena teks langkah masaknya sendiri mengandung tanda kutip `'`/`"` — ini kolom `steps` saja yang hanya dipakai untuk ditampilkan ke user, bukan untuk embedding/matching, jadi cukup aman diabaikan).

📌 **Untuk export CSV yang benar di masa depan** (kalau butuh lagi), ganti cell terakhir di `preprocessing.ipynb` jadi:
```python
import json
df_csv = df.copy()
list_cols = ['ingredients_clean', 'ingredients_parsed', 'ingredients_name_only', 'steps']
for col in list_cols:
    df_csv[col] = df_csv[col].apply(lambda x: json.dumps(list(x), ensure_ascii=False))
df_csv.to_csv('dataset_resep_preprocessed_5000.csv', index=False)
# waktu load ulang: df[col] = df[col].apply(json.loads)
```

**2) Model `all-indo-e5-small-v4` itu model keluarga E5 → WAJIB pakai prefix**

Model ini adalah hasil fine-tune dari `intfloat/multilingual-e5-small` (lihat model card LazarusNLP). Semua model E5 **butuh prefix di depan teks** supaya hasil embedding-nya optimal untuk retrieval asimetris (query pendek vs dokumen panjang):
- Teks resep (dokumen) → diberi prefix **`"passage: "`**
- Input bahan dari user (query) → diberi prefix **`"query: "`**

Kalau prefix ini dilewatkan, embedding tetap jalan (tidak error) tapi similarity score-nya akan lebih buruk — jadi penting untuk skripsi supaya akurasi retrieval-nya optimal.

In [ ]:
import pandas as pd
import numpy as np
import ast
import re
import json
import time
from pathlib import Path

pd.set_option("display.max_colwidth", 150)

# ==== KONFIGURASI ====
DATA_DIR = Path(".")  # ganti sesuai lokasi file kamu
PARQUET_PATH = DATA_DIR / "dataset_resep_preprocessed_5000.parquet"

MODEL_NAME = "LazarusNLP/all-indo-e5-small-v4"
CHROMA_DIR = "./chroma_db"
COLLECTION_NAME = "resep_masakan"

EMBED_BATCH_SIZE = 64  # sesuaikan dengan RAM/VRAM kamu
CHROMA_ADD_BATCH_SIZE = 500

LIST_COLS = [
    "ingredients_clean",
    "ingredients_parsed",
    "ingredients_name_only",
    "steps",
]

## 1. Load Dataset

Utamakan load dari `.parquet` (paling aman). Kalau file parquet tidak ada, otomatis fallback ke CSV + perbaikan bug.

In [ ]:
def repair_numpy_list_str(s):
    """
    Perbaikan untuk bug numpy-array-str -> CSV (lihat penjelasan di atas).
    Menyisipkan koma di antara elemen sebelum di-parse dengan ast.literal_eval.
    """
    if not isinstance(s, str):
        return s
    fixed = re.sub(r"'\s+'", "', '", s)  # antar string dg quote tunggal
    fixed = re.sub(r"}\s*\n?\s*{", "}, {", fixed)  # antar dict (ingredients_parsed)
    fixed = re.sub(r"'\s*\n\s*'", "', '", fixed)  # newline antar string
    return ast.literal_eval(fixed)


def load_dataset():
    print(f"Load dari parquet: {PARQUET_PATH}")
    df = pd.read_parquet(PARQUET_PATH)
    # pastikan kolom list bertipe python list (bukan numpy array) biar konsisten
    for col in LIST_COLS:
        df[col] = df[col].apply(lambda x: list(x) if isinstance(x, np.ndarray) else x)
    return df


df = load_dataset()
print("Shape:", df.shape)
print(df.dtypes)
df.head(2)

Load dari parquet: dataset_resep_preprocessed_5000.parquet
Shape: (5000, 10)
title                    object
ingredients_clean        object
ingredients_text         object
ingredients_parsed       object
ingredients_name_only    object
steps                    object
embedding_text           object
num_ingredients           int64
num_steps                 int64
char_count                int64
dtype: object


,title,ingredients_clean,ingredients_text,ingredients_parsed,ingredients_name_only,steps,embedding_text,num_ingredients,num_steps,char_count
0,Ikan Suwir Pedas,"[1 ekor Baby Tuna, 1 sdm Asam Jawa, 1 sdt Garam, 1/2 sdt Kunyit Bubuk, 500 ml Air, 1 lembar Daun Salam, 1 lembar Daun Jeruk, 1 genggam Daun Kemang...","1 ekor Baby Tuna, 1 sdm Asam Jawa, 1 sdt Garam, 1/2 sdt Kunyit Bubuk, 500 ml Air, 1 lembar Daun Salam, 1 lembar Daun Jeruk, 1 genggam Daun Kemangi...","[{'raw': '1 ekor Baby Tuna', 'quantity': '1', 'unit': 'ekor', 'name': 'baby tuna'}, {'raw': '1 sdm Asam Jawa', 'quantity': '1', 'unit': 'sdm', 'na...","[baby tuna, asam jawa, garam, kunyit bubuk, air, daun salam, daun jeruk, daun kemangi, bawang merah, bawang putih, cabe merah besar, cabe rawit, j...","[Bersihkan ikan, campur dengan air, asam jawa, garam dan kunyit, kemudian masak hingga ikan matang., Ambil bagian dagingnya kemudian suwir suwir, ...","Ikan Suwir Pedas. Bahan: 1 ekor Baby Tuna, 1 sdm Asam Jawa, 1 sdt Garam, 1/2 sdt Kunyit Bubuk, 500 ml Air, 1 lembar Daun Salam, 1 lembar Daun Jeru...",16,5,696
1,Minyak Ayam Bawang,"[100 g kulit ayam, 12 siung bawang merah, 4 siung bawang putih, 250 ml minyak goreng]","100 g kulit ayam, 12 siung bawang merah, 4 siung bawang putih, 250 ml minyak goreng","[{'raw': '100 g kulit ayam', 'quantity': '100', 'unit': 'g', 'name': 'kulit ayam'}, {'raw': '12 siung bawang merah', 'quantity': '12', 'unit': 'si...","[kulit ayam, bawang merah, bawang putih, minyak goreng]","[Rebus kulit ayam hingga air mendidih, angkat., Blender kulit ayam rebus, bawang merah, dan bawang putih. Tambahkan sedikit minyak sebagai pelarut...","Minyak Ayam Bawang. Bahan: 100 g kulit ayam, 12 siung bawang merah, 4 siung bawang putih, 250 ml minyak goreng",4,4,387


In [3]:
# Validasi cepat: pastikan hasil load sudah benar (list beneran, bukan 1 string gabungan)
sample = df.iloc[0]
print("ingredients_name_only (contoh):", sample["ingredients_name_only"])
print(
    "jumlah item:",
    len(sample["ingredients_name_only"]),
    "| num_ingredients:",
    sample["num_ingredients"],
)
assert len(sample["ingredients_name_only"]) > 1, (
    "Masih ke-merge jadi 1 string! Cek ulang loader."
)

# cross-check semua baris: panjang ingredients_clean harus = num_ingredients
mismatch = (df["ingredients_clean"].apply(len) != df["num_ingredients"]).sum()
print(
    f"Baris dengan ingredients_clean tidak cocok num_ingredients: {mismatch} / {len(df)}"
)

ingredients_name_only (contoh): ['baby tuna', 'asam jawa', 'garam', 'kunyit bubuk', 'air', 'daun salam', 'daun jeruk', 'daun kemangi', 'bawang merah', 'bawang putih', 'cabe merah besar', 'cabe rawit', 'jahe', 'serai', 'garam', 'kaldu bubuk']
jumlah item: 16 | num_ingredients: 16
Baris dengan ingredients_clean tidak cocok num_ingredients: 0 / 5000


## 2. Load Model SBERT

In [4]:
# Jalankan sekali saja kalau package belum ke-install
# !pip install -q sentence-transformers chromadb tqdm

In [5]:
import torch
from sentence_transformers import SentenceTransformer

device = (
    "cuda"
    if torch.cuda.is_available()
    else ("mps" if torch.backends.mps.is_available() else "cpu")
)
print("Device:", device)

model = SentenceTransformer(MODEL_NAME, device=device)
print("Embedding dimension:", model.get_sentence_embedding_dimension())

c:\Users\Mayceren\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4738.06it/s]


Embedding dimension: 384


C:\Users\Mayceren\AppData\Local\Temp\ipykernel_24248\2401881782.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", model.get_sentence_embedding_dimension())


## 3. Encoding dengan Prefix E5

`embedding_text` (title + bahan) diberi prefix `"passage: "` sebelum di-encode, sesuai konvensi model E5.

In [6]:
texts_to_encode = ("passage: " + df["embedding_text"]).tolist()
print("Contoh teks yang akan di-encode:\n", texts_to_encode[0][:200], "...")

start = time.time()
embeddings = model.encode(
    texts_to_encode,
    batch_size=EMBED_BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True,  # penting: biar cosine similarity = dot product
    convert_to_numpy=True,
)
elapsed = time.time() - start
print(f"Selesai encode {len(texts_to_encode)} resep dalam {elapsed:.1f} detik")
print("Shape embeddings:", embeddings.shape)

# checkpoint: simpan supaya tidak perlu encode ulang kalau notebook di-restart
np.save("embeddings_5000.npy", embeddings)

Contoh teks yang akan di-encode:
 passage: Ikan Suwir Pedas. Bahan: 1 ekor Baby Tuna, 1 sdm Asam Jawa, 1 sdt Garam, 1/2 sdt Kunyit Bubuk, 500 ml Air, 1 lembar Daun Salam, 1 lembar Daun Jeruk, 1 genggam Daun Kemangi, 3 buah Bawang Mera ...


Batches: 100%|██████████| 79/79 [01:33<00:00,  1.18s/it]

Selesai encode 5000 resep dalam 93.0 detik
Shape embeddings: (5000, 384)


## 4. Indexing ke ChromaDB

Metadata di ChromaDB hanya boleh berisi tipe scalar (`str`/`int`/`float`/`bool`), jadi kolom-kolom list/dict (`ingredients_clean`, `ingredients_parsed`, `ingredients_name_only`, `steps`) di-*JSON-encode* dulu jadi string sebelum disimpan sebagai metadata, dan di-`json.loads` balik waktu ditampilkan.

In [7]:
import chromadb

client = chromadb.PersistentClient(path=CHROMA_DIR)

# hapus collection lama kalau mau re-index dari nol (uncomment kalau perlu)
# client.delete_collection(COLLECTION_NAME)

collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)
print("Collection:", collection.name, "| jumlah item sekarang:", collection.count())

Collection: resep_masakan | jumlah item sekarang: 0


In [8]:
def build_metadata(row):
    return {
        "title": row["title"],
        "ingredients_clean": json.dumps(
            list(row["ingredients_clean"]), ensure_ascii=False
        ),
        "ingredients_parsed": json.dumps(
            list(row["ingredients_parsed"]), ensure_ascii=False
        ),
        "ingredients_name_only": json.dumps(
            list(row["ingredients_name_only"]), ensure_ascii=False
        ),
        "steps": json.dumps(list(row["steps"]), ensure_ascii=False),
        "num_ingredients": int(row["num_ingredients"]),
        "num_steps": int(row["num_steps"]),
    }


ids = [f"recipe_{i}" for i in range(len(df))]
documents = df["embedding_text"].tolist()
metadatas = [build_metadata(row) for _, row in df.iterrows()]
embeddings_list = embeddings.tolist()

n = len(df)
for start_idx in range(0, n, CHROMA_ADD_BATCH_SIZE):
    end_idx = min(start_idx + CHROMA_ADD_BATCH_SIZE, n)
    collection.add(
        ids=ids[start_idx:end_idx],
        embeddings=embeddings_list[start_idx:end_idx],
        documents=documents[start_idx:end_idx],
        metadatas=metadatas[start_idx:end_idx],
    )
    print(f"Indexed {end_idx}/{n}")

print("Total item di collection sekarang:", collection.count())

Indexed 500/5000
Indexed 1000/5000
Indexed 1500/5000
Indexed 2000/5000
Indexed 2500/5000
Indexed 3000/5000
Indexed 3500/5000
Indexed 4000/5000
Indexed 4500/5000
Indexed 5000/5000
Total item di collection sekarang: 5000


## 5. Uji Coba Retrieval

Query user diberi prefix `"query: "`. Ini baru retrieval murni berdasarkan similarity SBERT — reranking berbasis efisiensi bahan (berapa banyak bahan resep yang match dengan bahan user) belum diterapkan di sini, itu tahap selanjutnya.

In [9]:
def search_recipes(user_ingredients_text: str, top_k: int = 5):
    query_text = "query: " + user_ingredients_text
    query_embedding = model.encode(
        [query_text], normalize_embeddings=True, convert_to_numpy=True
    ).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k,
    )

    rows = []
    for i in range(len(results["ids"][0])):
        meta = results["metadatas"][0][i]
        rows.append(
            {
                "title": meta["title"],
                "similarity": 1
                - results["distances"][0][
                    i
                ],  # chroma cosine space -> distance = 1 - cosine_sim
                "ingredients_name_only": json.loads(meta["ingredients_name_only"]),
                "num_ingredients": meta["num_ingredients"],
            }
        )
    return pd.DataFrame(rows)


# contoh sesuai skenario di skripsi kamu: user cuma input bahan utama
contoh_query = "2 butir telur, 100gr dada ayam, 2 siung bawang putih"
hasil = search_recipes(contoh_query, top_k=5)
hasil

,title,similarity,ingredients_name_only,num_ingredients
0,Dadar telur,0.745163,"[telur, garam, micin, merica, bawang merah]",5
1,Dadar telur / omelet isi kornet dan bawang bombay,0.701970,"[telur ayam, bawang bombay, kornet, lada, penyedap rasa, garam, margarin 1 sdm dan minyak goreng 1 sdm]",7
2,Telur Dadar,0.682858,"[telur ayam, kornet sapi, bawang daun, keju cheddar, bawang putih bubuk, lada bubuk, garam]",7
3,Dadar telur kuah,0.673384,"[telur, tomat, daun bawang potong kecil"", helai daun sop, bawang putih cincang halus, minyak wijen, garam, penyedap , merica, air]",8
4,Telur Dadar Serundeng,0.669924,"[telur ayam, daun bawang, cabe merah keriting, terigu serbaguna, tepung beras, serundeng kelapa, garam, lada bubuk, air, minyak goreng]",10


## Catatan & Langkah Selanjutnya

1. **Reranking efisiensi bahan** — untuk tiap kandidat hasil retrieval SBERT di atas, hitung skor efisiensi berdasarkan `ingredients_name_only` vs bahan yang diketik user (contoh sederhana: `jumlah_bahan_match / total_bahan_resep`), lalu gabungkan (mis. weighted sum atau alpha-blend) dengan skor similarity SBERT untuk urutan akhir.
2. **Alur konfirmasi bahan pelengkap** — setelah retrieval awal, sebelum menampilkan hasil final, AI perlu bertanya balik ke user apakah ada bahan lain di luar yang sudah diketik (misal bumbu dapur standar: garam, minyak, bawang). Ini bisa diimplementasikan sebagai satu langkah percakapan terpisah (pakai Llama3) sebelum tahap reranking dijalankan — belum termasuk di notebook ini.
3. Kolom `ingredients_parsed` (hasil parsing quantity/unit/name) sudah tersimpan di metadata Chroma, jadi nanti bisa dipakai juga kalau butuh logika lebih detail dari sekadar nama bahan (misalnya cek jumlah/takaran).